In [24]:
import aspose.words_foss as aw 

docs_path = './docs/'
filename = "uk-rf.doc"

doc = aw.Document(docs_path+filename)
text = doc.get_text()

In [25]:
import numpy as np

def split_into_chunks(text: str, chunk_size: int=800,  overlap: int=80) -> list[str]:
    # для каждого чанка выведем начальный и конечный индексы, чтобы 
    if overlap >= chunk_size:
        raise ValueError("chunk_size должен быть больше overlap!")

    if overlap < 0:
        raise ValueError("overlap не должен быть меньше нуля!")

    if chunk_size < 0:
        raise ValueError("chunk_size не может быть меньше нуля!")

    if not text:
        raise ValueError("Текст пустой!")

    start_indexes = np.arange(0, len(text), chunk_size-overlap)
    end_indexes = start_indexes + chunk_size

    chunks = [
        text[start:end] for start, end in zip(start_indexes, end_indexes)
    ]

    return chunks

In [28]:
import torch
import torch.nn.functional as F
import polars as pl
import uuid
from transformers import AutoTokenizer, AutoModel
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

pl.Config.set_fmt_str_lengths(1000)
torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"

cache_dir = './models/'
embeding_model_name = "google/embeddinggemma-2"

vdb_path = './database/test'
vdb_filename = 'test.parquet'

tokenizer = AutoTokenizer.from_pretrained(embeding_model_name, cache_dir=cache_dir)
model = AutoModel.from_pretrained(embeding_model_name, cache_dir=cache_dir).to(device).eval()

client = QdrantClient(path=vdb_path)
collection_name = "base"

if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=model.config.text_config.embedding_dim,
            distance=models.Distance.COSINE
        )
    )

# Перешёл на langchain
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=60)
chunks = text_splitter.split_text(text)

batch_size = 128
for i in np.arange(0, len(chunks), batch_size):
    ids = tokenizer(
        chunks[i:i+batch_size],
        return_tensors="pt",
        padding=True
    )

    batch = {
        key: value.to(device)
        for key, value in ids.items()
    }

    with torch.inference_mode():
        output = model(**batch).last_hidden_state

    output = (output * batch['attention_mask'].unsqueeze(-1)).sum(1) / batch['attention_mask'].sum(1, keepdim=True)
    output = F.normalize(output, dim=1)

    client.upsert(
        collection_name=collection_name,
        points=[
            models.PointStruct(
                id=str(
                    uuid.uuid5(
                        uuid.NAMESPACE_URL,
                        f"{filename}:{idx}"
                    )
                ),
                vector=vector.cpu().to(torch.float64).numpy(),
                payload={
                    'text': text,
                    'doc': filename
                }
            ) for idx, vector, text in zip(range(i, i+batch_size), output, chunks[i:i+batch_size])
        ]
    )

del output;

Loading weights:   0%|          | 0/1376 [00:00<?, ?it/s]

In [ ]:
# search in base
query1 = "Мне нужна узнать: что такое кража?"
query2 = "В какой статье есть наказание за убийство?"
query3 = "Какой кодекс в арбитражном суде?"

In [ ]:
ids = tokenizer(
    query2, 
    attention_mask=True,
    return_tensors="pt"
).to(device)

attention_mask = ids['attention_mask']
output = (model(**ids).last_hidden_state * attention_mask.unsqueeze(-1)).sum(1) / attention_mask.sum(1, keepdim=True)
output = F.normalize(output, dim=1)
output = output.to(torch.float64).cpu()

In [ ]:
def cosine_similarity(x, y):
    return x @ y

db = pl.read_parquet(vdb_path+vdb_filename)
vector_db = torch.tensor(db['embedding_vector'].to_list(), dtype=torch.float64)
top_k_scores, top_k_indices = torch.topk(vector_db @ output.squeeze(0), k=40)

result = (
    db[top_k_indices.cpu().numpy()]
)

In [ ]:
# rerank
from sentence_transformers import CrossEncoder

reranker_name = "Qwen/Qwen3-Reranker-0.6B"
reranking_model = CrossEncoder(reranker_name, cache_folder=cache_dir, device=device)

pairs = [(query2, doc) for doc in result['text']]
scores = reranking_model.predict(pairs)

del reranking_model; del model; del output

In [ ]:
k=5
indexes = np.argsort(scores, descending=True)[:k]
context_top_k = result[indexes]['text'].to_list()
context_top_k

In [ ]:
from transformers import AutoModelForCausalLM

# Generate answer
k=5
indexes = np.argsort(scores, descending=True)[:k]
context_top_k = result[indexes]['text'].to_list()
context = '---'.join(context_top_k)

prompt = f"""
Ты отвечаешь на вопрос только по переданному контексту.

Правила:
- Используй только факты, прямо указанные в контексте
- Не придумывай номера статей, наказания, определения, примеры и обстоятельства
- Не объединяй сведения из разных статей, если контекст прямо не связывает их
- Если информации недостаточно, напиши: «В переданном контексте нет достаточной информации для ответа».
- Не давай юридических советов и не утверждай то, чего нет в контексте
- Ответ должен быть кратким, точным и законченным
- Ответ не должен превышать 300 слов
- Обязательно закончи все разделы в пределах этого ограничения
- Не начинай новый пункт, если не можешь закончить его в лимите

Структура ответа:

## Краткий ответ
...

## Подтверждение из контекста
- **[номер статьи или заголовок, если он есть в контексте]**: что именно сказано.
- **[номер статьи или заголовок, если он есть в контексте]**: что именно сказано.

## Ограничения
Что нельзя установить только по переданным фрагментам.

Контекст:
---
{context}
---

Вопрос: {query2}

Ответ:
"""

generative_model_name = "Qwen/Qwen3-4B-Instruct-2507"

gen_tokenizer = AutoTokenizer.from_pretrained(
    generative_model_name,
    cache_dir=cache_dir,
)

gen_model = AutoModelForCausalLM.from_pretrained(
    generative_model_name,
    cache_dir=cache_dir
).to(device).eval()
gen_model.requires_grad_(False)

messages = [{"role": "user", "content": prompt}]

inputs = gen_tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(device)

with torch.inference_mode():
    generated = gen_model.generate(
        **inputs,
        max_new_tokens=2048,
    )

answer = gen_tokenizer.decode(
    generated[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print(answer)